# 🌊 Groundwater Level Prediction Pipeline & Model
A clean, modular, and production-ready machine learning pipeline for predicting groundwater levels:
- **Custom Scikit-Learn Data Preprocessor (`GroundwaterDataPreprocessor`)**
- **Automated Feature Engineering** (Temporal, Cyclical Seasonality, and Station Lags)
- **Hyperparameter Tuning (`RandomizedSearchCV`)**
- **Unified Scikit-Learn `Pipeline`** (Raw Data $\rightarrow$ Features $\rightarrow$ Predictions)
- **Evaluation Plots & Zero-Friction CSV Inference**

In [ ]:
# 1. Setup & Imports
%pip install -q xgboost scikit-learn pandas pyarrow matplotlib seaborn

import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 2. Custom Scikit-Learn Data Pipeline Transformer

In [ ]:
class GroundwaterDataPreprocessor(BaseEstimator, TransformerMixin):
    """
    Automated data transformation pipeline for groundwater datasets:
    - Decomposes 'Data Acquisition Time' into Year, Month, Sin_Month, Cos_Month
    - Computes/imputes Station-level historical lag readings (Last_GWL)
    - Encodes categorical columns (District, Tehsil, Block, Station)
    """
    def __init__(self, cat_cols=None, num_cols=None, target_col='Groundwater Level Quarterly Manual (meter)'):
        self.cat_cols = cat_cols if cat_cols else ['District', 'Tehsil', 'Block', 'Station']
        self.num_cols = num_cols if num_cols else ['Latitude', 'Longitude', 'Year', 'Sin_Month', 'Cos_Month', 'Last_GWL']
        self.target_col = target_col
        self.features = self.cat_cols + self.num_cols
        self.encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        self.station_history_lookup = {}
        self.global_median_gwl = 5.0

    def fit(self, X, y=None):
        X_df = X.copy()
        self.encoder.fit(X_df[self.cat_cols].astype(str))
        
        if y is not None:
            X_df[self.target_col] = y
            
        if self.target_col in X_df.columns:
            valid_df = X_df.dropna(subset=[self.target_col])
            if 'Data Acquisition Time' in valid_df.columns:
                valid_df = valid_df.sort_values(by=['Station', 'Data Acquisition Time'])
            self.station_history_lookup = valid_df.groupby('Station')[self.target_col].last().to_dict()
            self.global_median_gwl = float(valid_df[self.target_col].median())
        return self

    def transform(self, X):
        X_df = X.copy()
        
        # 1. Temporal & Cyclical Feature Engineering
        if 'Data Acquisition Time' in X_df.columns:
            dates = pd.to_datetime(X_df['Data Acquisition Time'], errors='coerce')
            X_df['Year'] = dates.dt.year.fillna(2021).astype(int)
            X_df['Month'] = dates.dt.month.fillna(1).astype(int)
        else:
            X_df['Year'] = X_df.get('Year', 2021)
            X_df['Month'] = X_df.get('Month', 1)

        X_df['Sin_Month'] = np.sin(2 * np.pi * X_df['Month'] / 12)
        X_df['Cos_Month'] = np.cos(2 * np.pi * X_df['Month'] / 12)
        
        # 2. Historical Lag Handling
        if 'Last_GWL' not in X_df.columns or X_df['Last_GWL'].isnull().any():
            if 'Station' in X_df.columns:
                mapped_lags = X_df['Station'].map(self.station_history_lookup).fillna(self.global_median_gwl)
                X_df['Last_GWL'] = X_df['Last_GWL'].fillna(mapped_lags) if 'Last_GWL' in X_df.columns else mapped_lags
            else:
                X_df['Last_GWL'] = self.global_median_gwl
                
        # 3. Categorical Encoding
        X_df[self.cat_cols] = self.encoder.transform(X_df[self.cat_cols].astype(str))
        
        return X_df[self.features].values

print("✅ GroundwaterDataPreprocessor defined successfully.")

## 3. Data Ingestion & Train-Test Split

In [ ]:
# Load Parquet Data
parquet_path = 'gwl_manual_quarterly_cgwb_hr_1991_2020.parquet'
raw_df = pd.read_parquet(parquet_path)
target_col = 'Groundwater Level Quarterly Manual (meter)'

# Clean and calculate training lags
df_clean = raw_df.dropna(subset=[target_col]).sort_values(by=['Station', 'Data Acquisition Time']).reset_index(drop=True)
df_clean['Last_GWL'] = df_clean.groupby('Station')[target_col].shift(1)
df_train_full = df_clean.dropna(subset=['Last_GWL']).reset_index(drop=True)

X_raw = df_train_full.drop(columns=[target_col])
y_raw = df_train_full[target_col]

X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)
print(f"Training samples: {len(X_train):,}, Testing samples: {len(X_test):,}")

## 4. Build, Tune & Train Unified Pipeline (Preprocessor + XGBoost)

In [ ]:
# Best tuned hyperparameters
best_params = {
    'n_estimators': 150,
    'max_depth': 8,
    'learning_rate': 0.03,
    'subsample': 0.7,
    'colsample_bytree': 0.85,
    'reg_alpha': 0.1,
    'reg_lambda': 2.0,
    'random_state': 42,
    'n_jobs': -1,
    'objective': 'reg:squarederror'
}

# Create Unified Scikit-Learn Pipeline
pipeline = Pipeline([
    ('preprocessor', GroundwaterDataPreprocessor()),
    ('model', xgb.XGBRegressor(**best_params))
])

# Fit pipeline directly on raw feature DataFrame
print("Fitting unified end-to-end pipeline...")
pipeline.fit(X_train, y_train)

# Predict directly on raw test DataFrame
y_pred = pipeline.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\n🏆 Pipeline Performance on Raw Test Data:")
print(f"   R² Score:   {r2:.4f} ({r2*100:.2f}%)")
print(f"   Loss (MSE): {mse:.4f}")
print(f"   RMSE:       {rmse:.4f} meters")
print(f"   MAE:        {mae:.4f} meters")

## 5. Evaluation Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.3, s=15, color='#02818a', label=f'Pipeline Predictions (R² = {r2:.3f})')
min_val, max_val = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal 1:1 Fit')
axes[0].set_title('Actual vs Predicted Groundwater Level', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Actual GWL (m)')
axes[0].set_ylabel('Predicted GWL (m)')
axes[0].legend(frameon=True)

# 2. Residual Distribution
residuals = y_test - y_pred
sns.histplot(residuals, kde=True, ax=axes[1], color='#67a9cf', bins=40)
axes[1].axvline(0, color='red', linestyle='--', lw=1.5, label='Zero Error')
axes[1].axvline(residuals.mean(), color='orange', linestyle=':', lw=1.5, label=f'Mean Error ({residuals.mean():.2f}m)')
axes[1].set_title(f'Residual Error Distribution (MAE: {mae:.2f}m)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Residual (Actual - Predicted) [m]')
axes[1].set_ylabel('Frequency')
axes[1].legend(frameon=True)

plt.tight_layout()
plt.show()

## 6. Export Unified Pipeline & Direct CSV Inference

In [ ]:
# Export the entire Pipeline object directly to Pickle
pipeline_pkl_path = 'unified_groundwater_pipeline.pkl'
with open(pipeline_pkl_path, 'wb') as f:
    pickle.dump(pipeline, f)
print(f"✅ Exported complete end-to-end Pipeline to: {pipeline_pkl_path}")

# Load and Predict directly on a raw CSV file in 2 lines
with open(pipeline_pkl_path, 'rb') as f:
    loaded_pipeline = pickle.load(f)

raw_csv_sample = pd.read_csv('input_sample_to_predict.csv')
pred = loaded_pipeline.predict(raw_csv_sample)

print(f"🎯 Prediction on Raw CSV Input: {pred[0]:.2f} meters")